In [0]:
%python
dbutils.widgets.text("ROOT_PATH", "s3://amzn-3-transactions-etl")

In [0]:
%python

root_path = dbutils.widgets.get("ROOT_PATH")

users_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{root_path}/landing/users/users_dirty.csv")

display(users_df.limit(2))

In [0]:
%python
root_path = dbutils.widgets.get("ROOT_PATH")

users_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        f"{root_path}/bronze/users/"
    ) \
        .saveAsTable("transactions_etl.bronze.users")

In [0]:
%python
from pyspark.sql.functions import col, to_date, regexp_replace

users_clean = users_df.select(
    col("user_id"), 
    col("first_name"),
    col("last_name"),
    col("email"),
    to_date(
        regexp_replace(col("signup_date"), r"\.", "/"),
        "M/d/yy"
    ).alias("signup_date"),
    col("country"),
    col("referral_source")
).dropDuplicates(["user_id"])

display(users_clean.limit(5))

In [0]:
%python
root_path = dbutils.widgets.get("ROOT_PATH")

users_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        f"{root_path}/silver/users/"
    ) \
        .saveAsTable("transactions_etl.silver.users")

In [0]:
%python
from pyspark.sql.functions import date_format, count, countDistinct

insights_gold = spark.table("transactions_etl.silver.users") \
    .withColumn("day_name", date_format(col("signup_date"), "EEEE")) \
    .groupBy("day_name", "referral_source") \
    .agg(
        count("*").alias("signups"),
        countDistinct("country").alias("unique_countries")
    ) \
    .orderBy("day_name", "referral_source")

display(insights_gold)

In [0]:
%python
root_path = dbutils.widgets.get("ROOT_PATH")

insights_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "path",
        f"{root_path}/gold/insights/"
    ) \
    .saveAsTable("transactions_etl.gold.insights")